# PornWorks Web Generator — Colab Prototype

Browser-based prompt-to-image interface for the PornWorks Real Porn V03 SDXL checkpoint.

**Important:** Run setup, then select **Runtime → Restart session**. After restart, run the remaining cells from top to bottom. For a first test, use 768×768, 30–35 steps, and a simple prompt.

Select **Runtime → Change runtime type → T4 GPU** if available. This notebook keeps the checkpoint's own scheduler and VAE settings; changing the VAE precision can cause decode artifacts. The temporary share link is accessible to anyone who has it while your runtime is active. Use only fictional consenting adults.

In [ ]:
# Remove incompatible optional quantization package and install versions compatible with this checkpoint
!pip uninstall -y torchao diffusers transformers
!pip -q install "diffusers==0.31.0" "transformers==4.44.2" "huggingface-hub<1.0" "accelerate==0.34.2" safetensors gradio

In [ ]:
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError("torchao is still installed. Run the setup cell again, then Runtime → Restart session.")

import torch
import gradio as gr
from diffusers import DiffusionPipeline

MODEL_ID = "John6666/pornworks-real-porn-v03-sdxl"
has_cuda = torch.cuda.is_available()
dtype = torch.float16 if has_cuda else torch.float32
print("GPU available:", has_cuda)
if has_cuda:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. CPU generation will be extremely slow; select a GPU runtime if possible.")

# Load the checkpoint's own scheduler and VAE configuration unchanged.
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, use_safetensors=True, safety_checker=None)
if has_cuda:
    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing()
else:
    pipe.to("cpu")
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print("Model loaded. Scheduler:", pipe.scheduler.__class__.__name__)
print("VAE dtype:", next(pipe.vae.parameters()).dtype)

In [ ]:
import random
import numpy as np

def generate_image(prompt, negative_prompt, width, height, steps, guidance, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Enter a prompt first.")
    if width % 8 or height % 8:
        raise gr.Error("Width and height must be divisible by 8.")
    actual_seed = int(seed)
    if actual_seed < 0:
        actual_seed = random.randint(0, 2**31 - 1)
    generator = torch.Generator(device="cuda" if has_cuda else "cpu").manual_seed(actual_seed)
    try:
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt or None,
                width=int(width),
                height=int(height),
                num_inference_steps=int(steps),
                guidance_scale=float(guidance),
                generator=generator,
                output_type="pil",
            )
        image = result.images[0].convert("RGB")
        arr = np.asarray(image)
        if arr.size == 0 or not np.isfinite(arr).all() or arr.std() < 1.0:
            raise gr.Error("The model returned a blank or invalid image. Check the Colab runtime and model download, then restart and retry.")
        # Save a local PNG so the interface serves a normal image file.
        path = f"/content/generated_{actual_seed}.png"
        image.save(path, format="PNG")
        return image, path, f"Seed: {actual_seed} | Size: {image.width}×{image.height}"
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")

custom_css = """
body { background: #101014; }
.gradio-container { max-width: 1100px !important; }
#hero { text-align: center; padding: 18px 0 8px; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"), css=custom_css, title="PornWorks Image Lab") as demo:
    gr.Markdown("# PornWorks Image Lab\nCreate photorealistic images with the Real Porn V03 SDXL model. Use fictional consenting adults only.", elem_id="hero")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", placeholder="Describe the fictional adult characters, scene, lighting, camera, and style…", lines=5)
            negative = gr.Textbox(label="Negative prompt", value="pixelated, low quality, blurry, jpeg artifacts, distorted anatomy, extra fingers, watermark, text", lines=2)
            with gr.Accordion("Generation settings", open=True):
                width = gr.Slider(512, 1024, value=768, step=64, label="Width")
                height = gr.Slider(512, 1024, value=768, step=64, label="Height")
                steps = gr.Slider(20, 50, value=32, step=1, label="Steps")
                guidance = gr.Slider(1, 10, value=5.0, step=0.5, label="Guidance scale")
                seed = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
            run = gr.Button("Generate image", variant="primary")
        with gr.Column(scale=1):
            output = gr.Image(label="Generated image", type="pil", format="png")
            file_output = gr.File(label="Download PNG")
            seed_output = gr.Textbox(label="Generation info", interactive=False)
    run.click(generate_image, inputs=[prompt, negative, width, height, steps, guidance, seed], outputs=[output, file_output, seed_output])
    gr.Markdown("**Privacy note:** This prototype does not intentionally save images to a database. Images are saved temporarily in the Colab runtime. Do not enter personal or confidential information.")

demo.queue(max_size=2).launch(share=True, debug=True)

## How to use
1. Run the setup cell and wait until it finishes.
2. Select **Runtime → Restart session**.
3. Run the remaining cells from top to bottom.
4. Confirm the printed scheduler is `EDMDPMSolverMultistepScheduler`.
5. Generate a simple test image first, such as: `a photorealistic still life of a red apple on a wooden table, natural window light, sharp focus`.
6. Use the image preview or the **Download PNG** button. If the image still looks corrupted, send the generation log and GPU name shown in the output.

The share link expires when the runtime stops.